# 🧬 Remedia — Yerel NVIDIA GPU Notebook

Bu notebook Linux/WSL2 veya NVIDIA Container Toolkit bulunan Docker ortamında çalışır.

**Önerilen ilk kurulum:**

```bash
bash scripts/setup_local.sh
```

Sonra bu notebook'u `Remedia Local (GPU)` kernel'iyle aç.

- Sonuçlar: `local_workspace/Remedia_results/`
- Pocket cache: `local_workspace/remedia_cache/pocket_cache.json`
- İlk denemede `GENERATE_N = 20` ve `ACCURACY_PROFILE = "balanced"` bırak.


In [ ]:
# 1 — Deney ayarları
UNIPROT_ID = "P00918"
BOX_DIM = 20

METHOD = "fusion"  # fusion | genetic | brics | random | pretrained
GENERATE_N = 20
GA_GENERATIONS = 3

DOCKING_MODE = "iki_asamali"  # iki_asamali | sadece_fast | sadece_accurate
ACCURACY_PROFILE = "balanced"  # balanced | final
TOP_FRACTION = 0.10

INSTALL_REINVENT4 = False
RUN_BENCHMARK = False
BENCHMARK_N = 3

USE_CACHED_POCKET = True
FORCE_REFRESH_POCKET = False

print("✅ Deney ayarları hazır")


In [ ]:
# 2 — Yerel ortamı ve araçları doğrula
import datetime
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [
    Path(os.environ["REMEDIA_HOME"]).resolve()
    if os.environ.get("REMEDIA_HOME") else None,
    cwd,
    cwd.parent,
]
REPO_DIR = next(
    (path for path in candidates if path and (path / "src").is_dir()),
    None,
)
if REPO_DIR is None:
    raise RuntimeError(
        "Remedia repo kökü bulunamadı. Notebook'u repo içinden aç veya "
        "REMEDIA_HOME değişkenini ayarla."
    )

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

WORKSPACE = Path(
    os.environ.get("REMEDIA_WORKSPACE", REPO_DIR / "local_workspace")
).resolve()
CACHE_DIR = WORKSPACE / "remedia_cache"
RESULTS_ROOT = WORKSPACE / "Remedia_results"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULTS_DIR = RESULTS_ROOT / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tool_root = REPO_DIR / ".remedia-tools"
gnina_candidates = [
    os.environ.get("GNINA_PATH"),
    shutil.which("gnina"),
    str(tool_root / "bin" / "gnina"),
    "/usr/local/bin/gnina",
]
GNINA_PATH = next(
    (str(Path(path)) for path in gnina_candidates if path and Path(path).exists()),
    None,
)

fpocket_candidates = [
    shutil.which("fpocket"),
    str(tool_root / "env" / "bin" / "fpocket"),
    "/opt/remedia-env/bin/fpocket",
]
FPOCKET_PATH = next(
    (str(Path(path)) for path in fpocket_candidates if path and Path(path).exists()),
    None,
)

if not GNINA_PATH or not FPOCKET_PATH:
    raise RuntimeError(
        "GNINA veya fpocket bulunamadı. Repo kökünde "
        "`bash scripts/setup_local.sh` çalıştır."
    )

os.environ["GNINA_PATH"] = GNINA_PATH
os.environ["PATH"] = (
    str(Path(FPOCKET_PATH).parent)
    + os.pathsep
    + os.environ.get("PATH", "")
)

gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError(
        "NVIDIA GPU görünmüyor. Sürücüyü/WSL2 GPU desteğini kontrol et; "
        "Docker kullanıyorsan konteyneri `--gpus all` ile başlat."
    )

print("✅ Yerel ortam hazır")
print(f"Repo: {REPO_DIR}")
print(f"GNINA: {GNINA_PATH}")
print(f"fpocket: {FPOCKET_PATH}")
print(f"Sonuç klasörü: {RESULTS_DIR}")


In [ ]:
# 3 — Hedef protein, pocket ve tohum moleküller
CACHE_KEY = UNIPROT_ID.strip().upper()
BOX_SIZE = (float(BOX_DIM),) * 3
POCKET_CACHE_PATH = CACHE_DIR / "pocket_cache.json"

try:
    pocket_cache = (
        json.loads(POCKET_CACHE_PATH.read_text())
        if POCKET_CACHE_PATH.exists()
        else {}
    )
except Exception:
    pocket_cache = {}

cached = (
    pocket_cache.get(CACHE_KEY)
    if USE_CACHED_POCKET and not FORCE_REFRESH_POCKET
    else None
)

import fetch_structure
import pocket_detection

pdb_path = fetch_structure.fetch_alphafold(CACHE_KEY)
RECEPTOR = str(pdb_path)

if cached:
    CENTER = tuple(float(value) for value in cached["center"])
    print(f"⚡ Pocket cache kullanıldı: {CENTER}")
else:
    pocket = pocket_detection.best_druggable_pocket(pdb_path)
    CENTER = tuple(round(float(value), 3) for value in pocket["center"])
    pocket_cache[CACHE_KEY] = {
        "center": list(CENTER),
        "source": "fpocket",
        "druggability": pocket.get("druggability"),
    }
    POCKET_CACHE_PATH.write_text(json.dumps(pocket_cache, indent=2))
    print(f"✅ fpocket merkezi: {CENTER}")

from known_ligands import fetch_known_ligands
from rdkit import Chem

ligands, message = fetch_known_ligands(CACHE_KEY, max_results=5)
print(message)
seeds = [item["smiles"] for item in ligands]
if not seeds:
    seeds = [
        "NS(=O)(=O)c1ccccc1",
        "CC(=O)Nc1nnc(s1)S(N)(=O)=O",
    ]
seeds = [smiles for smiles in seeds if Chem.MolFromSmiles(smiles)]
if not seeds:
    raise RuntimeError("Geçerli tohum molekül bulunamadı.")

print(f"✅ Hedef={CACHE_KEY} · center={CENTER} · seed={len(seeds)}")


In [ ]:
# 4 — Molekül üretimi ve batch GNINA docking
import pandas as pd

from molecule_generator import (
    brics_recombination,
    fusion_generation,
    genetic_algorithm,
    random_mutation,
    write_smi,
)

method = METHOD.lower().strip()
if method == "pretrained":
    if not INSTALL_REINVENT4:
        raise RuntimeError(
            "METHOD='pretrained' için INSTALL_REINVENT4=True yap."
        )
    from generative_model import install_reinvent, generate_with_reinvent

    install_reinvent(log_fn=print, drive_cache_dir=CACHE_DIR)
    generated = generate_with_reinvent(
        num_molecules=GENERATE_N,
        output_path=RESULTS_DIR / "generated.smi",
        drive_cache_dir=CACHE_DIR,
    )
elif method == "fusion":
    final, _ = fusion_generation(
        seeds,
        docking_opts=None,
        log_fn=print,
        population_size=max(10, GENERATE_N),
        generations=GA_GENERATIONS,
    )
    generated = [smiles for smiles, _score in final]
elif method == "genetic":
    final, _ = genetic_algorithm(
        seeds,
        generations=GA_GENERATIONS,
        population_size=max(10, GENERATE_N),
        docking_opts=None,
        log_fn=print,
    )
    generated = [smiles for smiles, _score in final]
elif method == "brics":
    generated = brics_recombination(seeds, n=GENERATE_N)
elif method == "random":
    generated = random_mutation(seeds, n=GENERATE_N)
else:
    raise ValueError(f"Bilinmeyen METHOD: {METHOD}")

if method != "pretrained" and len(generated) < GENERATE_N:
    pool = set(generated)
    pool.update(random_mutation(seeds, n=GENERATE_N))
    pool.update(brics_recombination(seeds, n=GENERATE_N))
    generated = list(pool)

generated = [smiles for smiles in generated if smiles][:GENERATE_N]
if not generated:
    raise RuntimeError("Molekül üretilemedi.")

molecules = [
    (f"mol_{index:03d}", smiles)
    for index, smiles in enumerate(generated, 1)
]
smi_out = RESULTS_DIR / "generated.smi"
if method != "pretrained":
    write_smi(generated, smi_out, prefix="mol")

import gnina_engine

common = dict(
    receptor=RECEPTOR,
    center=CENTER,
    size=BOX_SIZE,
    gnina_path=GNINA_PATH,
    out_dir=RESULTS_DIR / "gnina_out",
    log_fn=print,
    profile=ACCURACY_PROFILE,
)

if DOCKING_MODE == "iki_asamali":
    rows, stage_info = gnina_engine.run_two_stage_screening(
        molecules,
        top_fraction=TOP_FRACTION,
        **common,
    )
elif DOCKING_MODE == "sadece_fast":
    rows, stage_info = gnina_engine.run_single_mode_screening(
        molecules,
        mode="fast",
        **common,
    )
elif DOCKING_MODE == "sadece_accurate":
    rows, stage_info = gnina_engine.run_single_mode_screening(
        molecules,
        mode="accurate",
        **common,
    )
else:
    raise ValueError(f"Bilinmeyen DOCKING_MODE: {DOCKING_MODE}")

docking_df = pd.DataFrame(rows)
display(docking_df)
print("✅ GNINA süreç sayısı:", stage_info.get("gnina_processes"))

if RUN_BENCHMARK:
    bench_rows, summary = gnina_engine.benchmark_fast_vs_accurate(
        molecules[:BENCHMARK_N],
        receptor=RECEPTOR,
        center=CENTER,
        size=BOX_SIZE,
        gnina_path=GNINA_PATH,
        out_dir=RESULTS_DIR / "benchmark",
        profile=ACCURACY_PROFILE,
    )
    pd.DataFrame(bench_rows).to_csv(
        RESULTS_DIR / "benchmark.csv",
        index=False,
    )
    print(summary)
else:
    print("⏭️ Benchmark kapalı")


In [ ]:
# 5 — Drug-likeness filtresi, sıralama ve kayıt
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import FileLink, display

import admet_filter

admet_df = pd.DataFrame(
    [
        admet_filter.lipinski_veber_filter(smiles, name)
        for name, smiles in molecules
    ]
)

docking_csv = RESULTS_DIR / "docking_scores.csv"
admet_csv = RESULTS_DIR / "admet_results.csv"
ranked_csv = RESULTS_DIR / "final_ranking.csv"

docking_df.to_csv(docking_csv, index=False)
admet_df.to_csv(admet_csv, index=False)

subprocess.run(
    [
        sys.executable,
        str(SRC_DIR / "rank_report.py"),
        "--docking",
        str(docking_csv),
        "--admet",
        str(admet_csv),
        "--output",
        str(ranked_csv),
    ],
    check=True,
)

ranked_df = pd.read_csv(ranked_csv)
display(ranked_df.head(10))

archive = shutil.make_archive(
    str(RESULTS_DIR),
    "zip",
    root_dir=RESULTS_DIR,
)
print(f"✅ Tamamlandı: {RESULTS_DIR}")
display(FileLink(archive))
